In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Document Loading

In [2]:
def load_all_docs(directory_path):
    documents = []
    # We loop through extensions to use the right loader for each
    for ext in ["**/*.pdf", "**/*.txt"]:
        loader = DirectoryLoader(
            directory_path, 
            glob=ext, 
            loader_cls=PyPDFLoader if ext.endswith(".pdf") else TextLoader,
            show_progress=True
        )
        documents.extend(loader.load())
    return documents

docs = load_all_docs("./data")
print(f"Loaded {len(docs)} documents.")

100%|██████████| 5/5 [00:00<00:00, 2818.37it/s]

Loaded 6 documents.


# Chunking

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Define the splitter
# chunk_size: How many characters per chunk (1000 is a safe starting point)
# chunk_overlap: Ensures context from the end of one chunk is included at the start of the next
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)

# 2. Perform the split
chunks = text_splitter.split_documents(docs)

# 3. Verify
print(f"Original document count: {len(docs)}")
print(f"Total chunks created: {len(chunks)}")

# Inspect the first chunk to see what it looks like
print("\n--- Example Chunk ---")
print(chunks[0].page_content)

Original document count: 6
Total chunks created: 26

--- Example Chunk ---
+ 3 3  7 4 4 7 4 2 3 0 0
P h o n e
c o n t a c t @ c h a i t a n y a m a l a n i . c o m
E m a i l
P a r i s ,  Î l e - d e - F r a n c e
L o c a t i o n
C o n t a c t  D e t a i l s
L i n k e d I n
l i n k e d i n . c o m / i n / c h a i t a n y a - m a l a n i /
F e b  2 0 2 4  -  M a y  2 0 2 5
E c o l e  d ' I n g é n i e u r s  e n  I n f o r m a t i q u e  ( E P I T A ) ,  P a r i s
M a s t e r ’ s  i n  C o m p u t e r  S c i e n c e
A u g  2 0 1 8  -  M a y  2 0 2 2
U n i v e r s i t y  o f  M u m b a i ,  I n d i a
B a c h e l o r ’ s  i n  C o m p u t e r  E n g i n e e r i n g
E d u c a t i o n
E x p e r i e n c e
J u l y  2 0 2 5  -  J a n  2 0 2 6
S C O R ,  P a r i s
A u t o m a t i o n  E n g i n e e r
I m p l e m e n t e d  c o n t a i n e r i z a t i o n  o f  t h e  a p p l i c a t i o n  u s i n g  D o c k e r
B u i l t  J e n k i n s  p i p e l i n e s  f o r  C I / C D  a c r o s s  m u l t 

# Vector Database

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Initialize the embedding model
# This model runs entirely on your CPU and is very efficient
print("Generating embeddings... this might take a moment.")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create the Vector Store (FAISS)
# This converts your 'chunks' into a searchable database
vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector Store created successfully!")

Generating embeddings... this might take a moment.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector Store created successfully!


In [5]:
# Test query
query = "Do you have leadership experience?"
results = vector_store.similarity_search(query, k=2)

print(f"Results for: '{query}'")
for i, res in enumerate(results):
    print(f"\n--- Result {i+1} (Source: {res.metadata.get('source')}) ---")
    print(res.page_content)

Results for: 'Do you have leadership experience?'

--- Result 1 (Source: data/experience/software_engineer_zeus_learning.txt) ---
---

8. Key Learnings:

- Strong experience in full-stack development using Node.js and React
- Understanding of building scalable and modular frontend systems
- Practical exposure to both SQL (PostgreSQL) and NoSQL (DynamoDB)
- Experience in data migration and system evolution
- Initial experience in team leadership and ownership

---

Technologies:
Node.js, React, PostgreSQL, ORM, Amazon DynamoDB, JavaScript

---

Personal Note:
This experience helped me build a strong foundation in full-stack development and system design. I particularly enjoyed working on reusable components and solving challenges related to handling diverse question types within a single platform. Leading a small team and owning features end-to-end also helped me grow in terms of responsibility and decision-making.

--- Result 2 (Source: data/experience/software_engineer_zeus_learning.t